In [1]:
"""
RHK Live-Fehleransicht Runner (robust, Jupyter-tauglich)

Ziele:
- Startet App als Subprocess
- Live stdout+stderr in Konsole + Logfile
- Crash/Exit sofort sichtbar (inkl. letzter Traceback)
- Port-Check: zeigt, ob Server wirklich lauscht
- Kein Jupyter-Blocking: Start läuft im Hintergrund-Thread

Hinweise:
- GRADIO_DEBUG muss "0" oder "1" sein
- Wenn dein App-Script PORT/GRADIO_SERVER_* respektiert, reicht ENV.
"""

from __future__ import annotations

from pathlib import Path
import os, sys, subprocess, threading, time, re, socket, signal, webbrowser
from datetime import datetime
from collections import deque
from typing import Optional

# -------------------------
# Konfiguration
# -------------------------
APP = Path("rhk_app_web_master.py")   # ggf. anpassen
HOST = "127.0.0.1"
PORT = 7700

AUTO_OPEN_BROWSER = False   # wenn True: öffnet Browser nach erfolgreichem Listen
STARTUP_WAIT_SEC = 8.0      # wie lange wir auf "Port lauscht" warten
LOG_DIR = Path("run_logs")
LOG_DIR.mkdir(exist_ok=True)

if not APP.exists():
    raise FileNotFoundError(f"Nicht gefunden: {APP.resolve()}")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_PATH = (LOG_DIR / f"gradio_verbose_{APP.stem}_{HOST}_{PORT}_{ts}.log").resolve()

# -------------------------
# Env (max verbose)
# -------------------------
env = os.environ.copy()
env["PORT"] = str(PORT)
env["GRADIO_SERVER_PORT"] = str(PORT)
env["GRADIO_SERVER_NAME"] = HOST

env["PYTHONUNBUFFERED"] = "1"
env["PYTHONFAULTHANDLER"] = "1"
env["PYTHONASYNCIODEBUG"] = "1"
env["PYTHONWARNINGS"] = env.get("PYTHONWARNINGS", "default")

env["GRADIO_DEBUG"] = env.get("GRADIO_DEBUG", "1")  # "0" oder "1"
env["GRADIO_ANALYTICS_ENABLED"] = "false"
env["LOG_LEVEL"] = env.get("LOG_LEVEL", "DEBUG")
env["ANYIO_DEBUG"] = env.get("ANYIO_DEBUG", "1")

# -------------------------
# Traceback Parsing / Highlight
# -------------------------
TB_START = "Traceback (most recent call last):"
TB_END_RE = re.compile(r"(\bError\b|\bException\b|SystemExit|KeyboardInterrupt)\s*:?")

HIGHLIGHT_RE = re.compile(
    r"\b("
    r"Traceback|Exception|Error|ValueError|NameError|AttributeError|KeyError|TypeError|"
    r"UnboundLocalError|AssertionError|RuntimeError|ImportError|ModuleNotFoundError|"
    r"SyntaxError|IndentationError|gradio|anyio|asyncio"
    r")\b"
)

# -------------------------
# State
# -------------------------
_proc: Optional[subprocess.Popen] = None
_stop = threading.Event()

_last_logs = deque(maxlen=2000)
_last_traceback = deque(maxlen=800)

_tb_mode = False
_reader_thread: Optional[threading.Thread] = None


def _is_port_listening(host: str, port: int, timeout: float = 0.2) -> bool:
    """Return True if TCP connect succeeds."""
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


def _stream_reader(proc: subprocess.Popen):
    """Read combined stdout/stderr line-by-line, write to logfile and console."""
    global _tb_mode
    with LOG_PATH.open("w", encoding="utf-8", newline="\n") as lf:
        while not _stop.is_set():
            line = proc.stdout.readline() if proc.stdout is not None else ""  # type: ignore
            if not line:
                if proc.poll() is not None:
                    break
                time.sleep(0.02)
                continue

            lf.write(line)
            lf.flush()

            s = line.rstrip("\n")
            _last_logs.append(s)

            # Traceback start
            if TB_START in s:
                _tb_mode = True
                _last_traceback.clear()
                _last_traceback.append(s)
                print("\n" + s)
                continue

            # Traceback body
            if _tb_mode:
                _last_traceback.append(s)
                print(s)
                # Ende: sobald wir eine Error/Exception Zeile sehen (heuristisch)
                if TB_END_RE.search(s):
                    _tb_mode = False
                    print("")
                continue

            # Highlight important lines
            if HIGHLIGHT_RE.search(s):
                _last_traceback.append(s)
                print("\n!!! " + s)
            else:
                print(s)


def start_app():
    """Start the app subprocess and begin streaming logs."""
    global _proc, _reader_thread

    if _proc is not None and _proc.poll() is None:
        print(f"App läuft bereits: PID={_proc.pid}")
        print(f"URL: http://{HOST}:{PORT}")
        print(f"Log: {LOG_PATH}")
        return

    _stop.clear()
    _last_logs.clear()
    _last_traceback.clear()

    print("=== STARTE APP (ROBUST VERBOSE RUNNER) ===")
    print(f"APP:  {APP.resolve()}")
    print(f"URL:  http://{HOST}:{PORT}")
    print(f"LOG:  {LOG_PATH}")
    print(f"ENV:  GRADIO_DEBUG={env.get('GRADIO_DEBUG')}  PYTHONWARNINGS={env.get('PYTHONWARNINGS')}")
    print("")

    cmd = [sys.executable, "-u", "-X", "faulthandler", str(APP)]
    print("$ " + " ".join(cmd))
    print("")

    _proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
        cwd=str(Path.cwd()),
    )

    _reader_thread = threading.Thread(target=_stream_reader, args=(_proc,), daemon=True)
    _reader_thread.start()

    # Startup monitor: Port listening check + early exit detection
    t0 = time.time()
    listened = False
    while time.time() - t0 < STARTUP_WAIT_SEC:
        if _proc.poll() is not None:
            code = _proc.returncode
            print("\n=== APP BEENDET (früh) ===")
            print(f"Returncode: {code}")
            print(f"Log: {LOG_PATH}")
            print("\nLetzter Traceback/Fehlerpuffer:")
            show_last_traceback()
            return

        if _is_port_listening(HOST, PORT):
            listened = True
            break

        time.sleep(0.2)

    if listened:
        print("\n=== APP LISTENING ===")
        print(f"Öffne: http://{HOST}:{PORT}")
        print(f"Log:   {LOG_PATH}")
        if AUTO_OPEN_BROWSER:
            try:
                webbrowser.open(f"http://{HOST}:{PORT}")
            except Exception:
                pass
    else:
        print("\n=== WARNUNG: Port lauscht nicht (noch) ===")
        print("App könnte noch starten ODER bereits intern hängen/fehlschlagen.")
        print("Nutze show_last_logs()/show_last_traceback().")
        print(f"Log: {LOG_PATH}")


def stop_app(force: bool = False):
    """Stop the app subprocess."""
    global _proc
    _stop.set()

    if _proc is None:
        print("Kein Prozess aktiv.")
        return

    if _proc.poll() is None:
        try:
            if force:
                # Hard kill
                if os.name == "nt":
                    _proc.kill()
                else:
                    _proc.send_signal(signal.SIGKILL)
            else:
                _proc.terminate()
            _proc.wait(timeout=5)
        except Exception:
            try:
                _proc.kill()
            except Exception:
                pass

    print(f"\n=== GESTOPPT ===\nLog: {LOG_PATH}")
    _proc = None


def show_last_traceback():
    """Print buffered traceback / highlighted errors."""
    if not _last_traceback:
        print("(Kein Traceback/Fehler im Puffer.)")
        return
    print("\n--- LETZTER FEHLERBLOCK (gepuffert) ---")
    print("\n".join(_last_traceback))


def show_last_logs(n: int = 300):
    """Print last n log lines."""
    if not _last_logs:
        print("(Keine Logs im Puffer.)")
        return
    n = max(1, min(n, len(_last_logs)))
    print(f"\n--- LETZTE {n} LOGZEILEN ---")
    print("\n".join(list(_last_logs)[-n:]))


def status():
    """Quick status helper."""
    if _proc is None:
        print("Status: kein Prozess.")
        return
    alive = _proc.poll() is None
    print(f"Status: PID={_proc.pid} alive={alive} returncode={_proc.returncode}")
    print(f"Port listening: {_is_port_listening(HOST, PORT)}")
    print(f"URL: http://{HOST}:{PORT}")
    print(f"Log: {LOG_PATH}")


# -------------
# Autostart
# -------------
if __name__ == "__main__":
    start_app()
    # Nutzung:
    # - stop_app() / stop_app(force=True)
    # - show_last_traceback()
    # - show_last_logs(500)
    # - status()


=== STARTE APP (ROBUST VERBOSE RUNNER) ===
APP:  C:\Users\Administrator\OneDrive\Forschung\Laienbefund-Projekt\RHK-BEfunder\rhk_app_web_master.py
URL:  http://127.0.0.1:7700
LOG:  C:\Users\Administrator\OneDrive\Forschung\Laienbefund-Projekt\RHK-BEfunder\run_logs\gradio_verbose_rhk_app_web_master_127.0.0.1_7700_20260209_175718.log
ENV:  GRADIO_DEBUG=1  PYTHONWARNINGS=default

$ C:\ProgramData\anaconda3\python.exe -u -X faulthandler rhk_app_web_master.py


!!! C:\ProgramData\anaconda3\Lib\asyncio\base_events.py:726: ResourceWarning: unclosed event loop <ProactorEventLoop running=False closed=False debug=True>
  _warn(f"unclosed event loop {self!r}", ResourceWarning, source=self)

!!! C:\ProgramData\anaconda3\Lib\asyncio\base_events.py:726: ResourceWarning: unclosed event loop <ProactorEventLoop running=False closed=False debug=True>
  _warn(f"unclosed event loop {self!r}", ResourceWarning, source=self)

!!! C:\ProgramData\anaconda3\Lib\asyncio\base_events.py:726: ResourceWarning: unclos